In [8]:

%load_ext autoreload
%autoreload 2

import pandas as pd
from optbinning import OptimalBinning
import numpy as np

train = pd.read_parquet("../data/processed/train.parquet")

var = "EXT_SOURCE_3"
optb = OptimalBinning(name=var, dtype="numerical", monotonic_trend="auto")
optb.fit(train[var], train["TARGET"])

optb.binning_table.build()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.18)",10885,0.050567,8509,2376,0.218282,-1.156807,0.108429,0.012845
1,"[0.18, 0.31)",20486,0.095170,17582,2904,0.141755,-0.631727,0.049442,0.006080
2,"[0.31, 0.37)",11009,0.051144,9893,1116,0.101372,-0.250437,0.003563,0.000444
3,"[0.37, 0.41)",10897,0.050623,9843,1054,0.096724,-0.198345,0.002164,0.000270
4,"[0.41, 0.46)",11364,0.052793,10477,887,0.078054,0.036579,0.000070,0.000009
5,"[0.46, 0.50)",12169,0.056532,11309,860,0.070671,0.143908,0.001102,0.000138
6,"[0.50, 0.57)",20527,0.095360,19318,1209,0.058898,0.33873,0.009501,0.001182
7,"[0.57, 0.64)",22025,0.102320,20957,1068,0.048490,0.544172,0.024187,0.002987
8,"[0.64, 0.70)",21536,0.100048,20671,865,0.040165,0.741244,0.040534,0.004954
9,"[0.70, 0.76)",17746,0.082441,17130,616,0.034712,0.892826,0.045640,0.005523


In [2]:
import sys
print(sys.executable)

/opt/anaconda3/envs/credit-risk/bin/python


## Set up max_n_bins = 8

In [3]:
optb = OptimalBinning(
    name=var,
    dtype="numerical",
    monotonic_trend="auto",
    max_n_bins=8,         # at most 8 bins
    min_bin_size=0.05,    # each bin at least 5% of the data
)
optb.fit(train[var], train["TARGET"])

optb.binning_table.build()

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.18)",10885,0.050567,8509,2376,0.218282,-1.156807,0.108429,0.012845
1,"[0.18, 0.31)",20486,0.095170,17582,2904,0.141755,-0.631727,0.049442,0.006080
2,"[0.31, 0.41)",21906,0.101767,19736,2170,0.099060,-0.224796,0.005651,0.000705
3,"[0.41, 0.50)",23533,0.109325,21786,1747,0.074236,0.090854,0.000869,0.000109
4,"[0.50, 0.57)",20527,0.095360,19318,1209,0.058898,0.33873,0.009501,0.001182
5,"[0.57, 0.64)",22025,0.102320,20957,1068,0.048490,0.544172,0.024187,0.002987
6,"[0.64, 0.70)",21536,0.100048,20671,865,0.040165,0.741244,0.040534,0.004954
7,"[0.70, inf)",31679,0.147168,30629,1050,0.033145,0.940644,0.088760,0.010703
8,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
9,Missing,42680,0.198275,38692,3988,0.093440,-0.16017,0.005440,0.000679


## Screen all variables in one loop

In [4]:
EXCLUDE = ["TARGET", "SK_ID_CURR"]
candidates = [c for c in train.columns if c not in EXCLUDE]

results = []
binners = {}
for var in candidates:
    is_num = pd.api.types.is_numeric_dtype(train[var])
    optb = OptimalBinning(
        name=var,
        dtype="numerical" if is_num else "categorical",
        monotonic_trend="auto" if is_num else None,
        max_n_bins=8,
        min_bin_size=0.05,
    )
    try:
        optb.fit(train[var], train["TARGET"])
        table = optb.binning_table.build()
        results.append({"variable": var, "iv": optb.binning_table.iv,
                        "n_bins": len(table) - 3, "status": optb.status})
        binners[var] = optb
    except Exception as e:
        results.append({"variable": var, "iv": None, "n_bins": None, "status": str(e)})

iv_table = pd.DataFrame(results).sort_values("iv", ascending=False)
iv_table.head(30)

,variable,iv,n_bins,status
38,EXT_SOURCE_3,0.332815,8,OPTIMAL
37,EXT_SOURCE_2,0.319643,8,OPTIMAL
36,EXT_SOURCE_1,0.150084,6,OPTIMAL
14,DAYS_EMPLOYED,0.116156,8,OPTIMAL
7,AMT_GOODS_PRICE,0.092841,8,OPTIMAL
13,DAYS_BIRTH,0.088861,8,OPTIMAL
75,AGE_YEARS,0.088861,8,OPTIMAL
24,OCCUPATION_TYPE,0.078018,5,OPTIMAL
74,CREDIT_GOODS_RATIO,0.073348,6,OPTIMAL
35,ORGANIZATION_TYPE,0.072242,8,OPTIMAL


### Check of expected affordability drivers

CREDIT_INCOME_RATIO, ANNUITY_INCOME_RATIO and AMT_INCOME_TOTAL were expected to be important affordability drivers (see expert long list), but have IV below 0.02. CREDIT_TERM passes the IV threshold (0.027) but got only 2 bins. Their binning tables are inspected below to understand why.

In [5]:
for var in ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM", "AMT_INCOME_TOTAL"]:
    print(var, iv_table.loc[iv_table["variable"] == var, "iv"].values)
    display(binners[var].binning_table.build())

CREDIT_INCOME_RATIO [0.01252876]


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 1.01)",11364,0.052793,10638,726,0.063886,0.252124,0.003021,0.000377
1,"[1.01, 1.53)",19016,0.088341,17585,1431,0.075252,0.076159,0.000496,0.000062
2,"[1.53, 2.00)",23112,0.107369,21296,1816,0.078574,0.02937,0.000091,0.000011
3,"[2.00, 2.27)",11263,0.052324,10323,940,0.083459,-0.036263,0.000070,0.000009
4,"[2.27, 2.50)",11064,0.051399,9994,1070,0.096710,-0.198187,0.002194,0.000274
5,"[2.50, 4.85)",78141,0.363013,71257,6884,0.088097,-0.09542,0.003440,0.000430
6,"[4.85, 8.99)",49955,0.232071,46214,3741,0.074887,0.081417,0.001487,0.000186
7,"[8.99, inf)",11342,0.052691,10573,769,0.067801,0.188455,0.001730,0.000216
8,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
9,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000


ANNUITY_INCOME_RATIO [0.00778653]


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.06)",11822,0.054920,11051,771,0.065217,0.230075,0.002641,0.000329
1,"[0.06, 0.11)",35149,0.163289,32558,2591,0.073715,0.098466,0.001519,0.000190
2,"[0.11, 0.16)",54979,0.255411,50695,4284,0.077921,0.038427,0.000371,0.000046
3,"[0.16, 0.20)",36586,0.169964,33493,3093,0.084541,-0.050318,0.000440,0.000055
4,"[0.20, 0.22)",20366,0.094612,18616,1750,0.085928,-0.068108,0.000452,0.000056
5,"[0.22, 0.27)",23314,0.108308,21236,2078,0.089131,-0.108221,0.001327,0.000166
6,"[0.27, 0.30)",10789,0.050121,9801,988,0.091575,-0.137956,0.001011,0.000126
7,"[0.30, inf)",22244,0.103337,20422,1822,0.081910,-0.015835,0.000026,0.000003
8,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
9,Missing,8,0.000037,8,0,0.000000,0.0,0.000000,0.000000


CREDIT_TERM [0.02714131]


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 10.78)",22797,0.105906,21689,1108,0.048603,0.541735,0.024836,0.003067
1,"[10.78, inf)",192452,0.894057,176183,16269,0.084535,-0.050252,0.002306,0.000288
2,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
3,Missing,8,0.000037,8,0,0.000000,0.0,0.000000,0.000000
Totals,,215257,1.000000,197880,17377,0.080727,,0.027141,0.003355


AMT_INCOME_TOTAL [0.01287805]


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 76374.00)",16648,0.077340,15311,1337,0.080310,0.00563,2.445604e-06,3.057001e-07
1,"[76374.00, 98977.50)",25332,0.117683,23216,2116,0.083531,-0.037199,1.654068e-04,2.067466e-05
2,"[98977.50, 127530.00)",36336,0.168803,33133,3203,0.088149,-0.096071,1.622093e-03,2.026837e-04
3,"[127530.00, 180045.00)",72423,0.336449,66276,6147,0.084876,-0.05465,1.028134e-03,1.285007e-04
4,"[180045.00, 211716.00)",15587,0.072411,14325,1262,0.080965,-0.003205,7.447138e-07,9.308918e-08
5,"[211716.00, 252225.00)",21537,0.100052,19933,1604,0.074476,0.087363,7.361932e-04,9.199490e-05
6,"[252225.00, 297607.50)",11475,0.053308,10704,771,0.067190,0.198171,1.927094e-03,2.404934e-04
7,"[297607.50, inf)",15919,0.073953,14982,937,0.058860,0.339408,7.395943e-03,9.200808e-04
8,Special,0,0.000000,0,0,0.000000,0.0,0.000000e+00,0.000000e+00
9,Missing,0,0.000000,0,0,0.000000,0.0,0.000000e+00,0.000000e+00


**Findings**
- CREDIT_INCOME_RATIO (IV 0.013): non-monotonic, with risk peaking at ratios 2.3–2.5 and lower at both ends. Likely selection bias: high-ratio loans were probably approved only for otherwise strong applicants. Dropped.
- ANNUITY_INCOME_RATIO (IV 0.008): risk rises as expected, from 6.5% for the lowest ratios to 9.2% at ratios 0.27–0.30, but falls back to 8.2% for the highest ratios. The overall effect is small, likely for the same selection reason. Dropped.
- CREDIT_TERM (IV 0.027): two bins only. Short loans (under about 11 payments, 11% of clients) default at 4.9%, versus 8.5% for longer loans. This confirms the expected effect "longer term → higher risk" and works as a simple, explainable split. Kept on the short list.
- AMT_INCOME_TOTAL (IV 0.013): the default rate is nearly flat (8.0–8.8%) up to an income of about 210,000, and only the highest incomes show lower risk (5.9% above about 300,000). Weak overall. Dropped.

The weak results for the affordability variables suggest that Home Credit's approval process already accounted for affordability, so among approved loans these variables add little. Affordability remains important in practice, but its effect cannot be measured well in data containing only approved applicants.

In [6]:
SELECTION_EXCLUDED = {
    "DAYS_BIRTH": "Duplicate of AGE_YEARS",
    "REGION_RATING_CLIENT": "Near-duplicate of REGION_RATING_CLIENT_W_CITY (lower IV)",
    "DAYS_EMPLOYED_ANOM": "Captured by the Missing bin of DAYS_EMPLOYED",
    "AMT_CREDIT": "Correlated with AMT_GOODS_PRICE (lower IV)",
}

shortlist = iv_table[
    (iv_table["iv"] >= 0.02) & (~iv_table["variable"].isin(SELECTION_EXCLUDED))
]

In [7]:
print(shortlist)

                       variable        iv  n_bins   status
38                 EXT_SOURCE_3  0.332815       8  OPTIMAL
37                 EXT_SOURCE_2  0.319643       8  OPTIMAL
36                 EXT_SOURCE_1  0.150084       6  OPTIMAL
14                DAYS_EMPLOYED  0.116156       8  OPTIMAL
7               AMT_GOODS_PRICE  0.092841       8  OPTIMAL
75                    AGE_YEARS  0.088861       8  OPTIMAL
24              OCCUPATION_TYPE  0.078018       5  OPTIMAL
74           CREDIT_GOODS_RATIO  0.073348       6  OPTIMAL
35            ORGANIZATION_TYPE  0.072242       8  OPTIMAL
8              NAME_INCOME_TYPE  0.054792       4  OPTIMAL
27  REGION_RATING_CLIENT_W_CITY  0.052011       3  OPTIMAL
9           NAME_EDUCATION_TYPE  0.048175       2  OPTIMAL
43       DAYS_LAST_PHONE_CHANGE  0.046210       8  OPTIMAL
16              DAYS_ID_PUBLISH  0.038663       8  OPTIMAL
12   REGION_POPULATION_RELATIVE  0.035504       7  OPTIMAL
19               FLAG_EMP_PHONE  0.032725       2  OPTIM

In [9]:
woe_train = pd.DataFrame({
    var: binners[var].transform(train[var], metric="woe")
    for var in shortlist["variable"]
})

corr = woe_train.corr().abs()
pairs = (
    corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    .stack()
    .sort_values(ascending=False)
)
pairs[pairs > 0.6]

NAME_INCOME_TYPE             FLAG_EMP_PHONE                0.784298
REGION_RATING_CLIENT_W_CITY  REGION_POPULATION_RELATIVE    0.704139
ORGANIZATION_TYPE            FLAG_EMP_PHONE                0.674237
                             NAME_INCOME_TYPE              0.636731
AGE_YEARS                    FLAG_EMP_PHONE                0.628143
dtype: float64

### Exclusions after the correlation check

Pairs of WoE-transformed variables with absolute correlation above 0.6 were reviewed. For each pair, the variable with the lower IV, or the weaker business meaning, is excluded:

- **FLAG_EMP_PHONE** appears in three pairs: with NAME_INCOME_TYPE (0.78), ORGANIZATION_TYPE (0.67) and AGE_YEARS (0.63). It mainly captures whether the client is employed or a pensioner, which other variables already cover, and has the lowest IV of the group.
- **REGION_POPULATION_RELATIVE** is correlated with REGION_RATING_CLIENT_W_CITY (0.70), which has the higher IV. Excluding it also addresses the fairness concern noted in the expert long list.
- **NAME_INCOME_TYPE** is correlated with ORGANIZATION_TYPE (0.64). ORGANIZATION_TYPE is kept for its higher IV (0.072 vs 0.055), subject to its bins making business sense.

In [10]:
SELECTION_EXCLUDED.update({
    "FLAG_EMP_PHONE": "Proxy for employed/pensioner; correlated with income type, organisation type and age (lowest IV)",
    "REGION_POPULATION_RELATIVE": "Correlated with REGION_RATING_CLIENT_W_CITY (0.70, lower IV); also a fairness concern",
    "NAME_INCOME_TYPE": "Correlated with ORGANIZATION_TYPE (0.64, lower IV)",
})

In [13]:
shortlist = iv_table[
    (iv_table["iv"] >= 0.02) & (~iv_table["variable"].isin(SELECTION_EXCLUDED))
]
woe_train = woe_train[shortlist["variable"]]

corr = woe_train.corr().abs()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print(len(shortlist), "variables")
print(pairs[pairs > 0.6])
print(shortlist)

21 variables
Series([], dtype: float64)
                       variable        iv  n_bins   status
38                 EXT_SOURCE_3  0.332815       8  OPTIMAL
37                 EXT_SOURCE_2  0.319643       8  OPTIMAL
36                 EXT_SOURCE_1  0.150084       6  OPTIMAL
14                DAYS_EMPLOYED  0.116156       8  OPTIMAL
7               AMT_GOODS_PRICE  0.092841       8  OPTIMAL
75                    AGE_YEARS  0.088861       8  OPTIMAL
24              OCCUPATION_TYPE  0.078018       5  OPTIMAL
74           CREDIT_GOODS_RATIO  0.073348       6  OPTIMAL
35            ORGANIZATION_TYPE  0.072242       8  OPTIMAL
27  REGION_RATING_CLIENT_W_CITY  0.052011       3  OPTIMAL
9           NAME_EDUCATION_TYPE  0.048175       2  OPTIMAL
43       DAYS_LAST_PHONE_CHANGE  0.046210       8  OPTIMAL
16              DAYS_ID_PUBLISH  0.038663       8  OPTIMAL
33       REG_CITY_NOT_WORK_CITY  0.032567       2  OPTIMAL
6                   AMT_ANNUITY  0.031555       8  OPTIMAL
15            DA

## Expected vs observed

| Variable | Expected | IV | Observed | Decision |
|---|---|---|---|---|
| EXT_SOURCE_3 | Higher → lower risk | 0.333 | Confirmed, monotonic | Keep |
| DAYS_EMPLOYED | Longer → lower risk | 0.116 | Confirmed, stronger after cleaning | Keep |
| CREDIT_INCOME_RATIO | Higher → higher risk | 0.013 | Weak, non-monotonic (peak at 2.3–2.5) | Drop |
| ... | | | | |

**Note on affordability ratios:** CREDIT_INCOME_RATIO was expected to be a key affordability driver but showed weak, non-monotonic predictive power. A likely reason is selection bias: the data contains only approved loans, and high-ratio loans were probably approved only for otherwise strong applicants.

**Choice of max_n_bins:** limiting EXT_SOURCE_3 to 8 bins reduced IV only from 0.3335 to 0.3328, while merging neighbouring bins with similar default rates. Fewer bins are simpler and more stable, so max_n_bins = 8 is used for all variables.

In [14]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

final_vars = shortlist["variable"].tolist()
joblib.dump({v: binners[v] for v in final_vars}, MODEL_DIR / "binners.joblib")
iv_table.to_csv(MODEL_DIR / "iv_table.csv", index=False)